# 10 · Mini Proyecto: Abandono de Clientes (Customer Churn)
**Curso ITEC · C-DAC Noida — Programa Especializado en IA**

Una empresa de telecomunicaciones quiere saber **qué clientes están por irse** (churn).
Vamos a entrenar una **red neuronal** que, con los datos de un cliente, predice si se queda o se va,
y además veremos **los motivos** más comunes del abandono.

Corre en Colab sin subir nada: el dataset se descarga solo desde internet.

### 1. Importar herramientas

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras

### 2. Cargar los datos
Usamos el dataset público **Telco Customer Churn** (unos 7.000 clientes).

In [2]:
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
print('Clientes:', df.shape[0], '· Columnas:', df.shape[1])
df.head()

Clientes: 7043 · Columnas: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 3. Limpiar y preparar
La columna TotalCharges trae algunos espacios en blanco: los quitamos y la pasamos a número.
Luego convertimos los textos (Sí/No) en 0/1 y las categorías en columnas con one-hot.

In [3]:
df = df.drop('customerID', axis='columns')
df = df[df.TotalCharges != ' '].copy()
df.TotalCharges = pd.to_numeric(df.TotalCharges)

# 'No internet service' y 'No phone service' cuentan como 'No'
df.replace('No internet service', 'No', inplace=True)
df.replace('No phone service', 'No', inplace=True)

# Sí/No -> 1/0
si_no = ['Partner','Dependents','PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
         'DeviceProtection','TechSupport','StreamingTV','StreamingMovies','PaperlessBilling','Churn']
for c in si_no:
    df[c] = df[c].replace({'Yes':1,'No':0})
df['gender'] = df['gender'].replace({'Female':1,'Male':0})

# One-hot de las categóricas
df = pd.get_dummies(data=df, columns=['InternetService','Contract','PaymentMethod'])
df = df.astype({c:int for c in df.columns if df[c].dtype=='bool'})
print('Columnas finales:', df.shape[1])

Columnas finales: 27


### 4. Escalar los números (0 a 1)
Las columnas grandes (tenure, cargos) se llevan al rango 0-1 para que la red aprenda mejor.

In [4]:
from sklearn.preprocessing import MinMaxScaler
cols = ['tenure','MonthlyCharges','TotalCharges']
df[cols] = MinMaxScaler().fit_transform(df[cols])

### 5. Separar entrenamiento y prueba
X = datos del cliente; y = si se fue (1) o se quedó (0).

In [5]:
from sklearn.model_selection import train_test_split
X = df.drop('Churn', axis='columns')
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5)
print('Entrenamiento:', X_train.shape, '· Prueba:', X_test.shape)

Entrenamiento: (5625, 26) · Prueba: (1407, 26)


### 6. Construir y entrenar la red neuronal
Una red sencilla: entrada → capa oculta (ReLU) → 1 neurona de salida **sigmoide**
(la salida es una probabilidad de 0 a 1: ¿se va o no?).

In [6]:
nf = X_train.shape[1]
model = keras.Sequential([
    keras.layers.Input(shape=(nf,)),
    keras.layers.Dense(nf, activation='relu'),
    keras.layers.Dense(15, activation='relu'),
    keras.layers.Dense(1,  activation='sigmoid')   # 1 = se va, 0 = se queda
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train.values.astype('float32'), y_train.values.astype('float32'), epochs=100, verbose=0)
loss, acc = model.evaluate(X_test.values.astype('float32'), y_test.values.astype('float32'), verbose=0)
print('Precisión en prueba: %.1f%%' % (acc*100))

Precisión en prueba: 77.4%


La red alcanza alrededor del **77%** de acierto al predecir quién se va.

In [7]:
from sklearn.metrics import classification_report, confusion_matrix
yp = model.predict(X_test.values.astype('float32'), verbose=0).flatten()
y_pred = (yp > 0.5).astype(int)
yt = y_test.values.astype(int)
print(classification_report(yt, y_pred, target_names=['Se queda (0)','Se va (1)']))

              precision    recall  f1-score   support

Se queda (0)       0.83      0.86      0.84       999
   Se va (1)       0.62      0.56      0.59       408

    accuracy                           0.77      1407
   macro avg       0.72      0.71      0.72      1407
weighted avg       0.77      0.77      0.77      1407



### 7. ¿Por qué se van los clientes? (los motivos)
Calculamos, para cada factor, qué porcentaje de esos clientes abandona.
Así la empresa sabe dónde actuar.

In [8]:
d = pd.read_csv(url)
d = d[d.TotalCharges != ' '].copy()
d['c'] = (d['Churn'] == 'Yes').astype(int)
promedio = d['c'].mean()*100
def tasa(mask): return d.loc[mask,'c'].mean()*100

motivos = {
 'Cliente nuevo (0-12 meses)': tasa(d.tenure <= 12),
 'Contrato mes a mes':         tasa(d.Contract == 'Month-to-month'),
 'Pago con cheque electrónico':tasa(d.PaymentMethod == 'Electronic check'),
 'Internet de fibra óptica':   tasa(d.InternetService == 'Fiber optic'),
 'Sin soporte técnico':        tasa(d.TechSupport == 'No'),
}
print('Abandono promedio general: %.0f%%\n' % promedio)
for k,v in sorted(motivos.items(), key=lambda x:-x[1]):
    print('%-32s %.0f%%' % (k, v))

Abandono promedio general: 27%

Cliente nuevo (0-12 meses)       48%
Pago con cheque electrónico      45%
Contrato mes a mes               43%
Internet de fibra óptica         42%
Sin soporte técnico              42%


### Resumen
- La red neuronal predice el abandono con **~77%** de acierto.
- Los que más se van: **clientes nuevos**, con **contrato mes a mes**, **pago por cheque electrónico**,
  **fibra óptica** y **sin soporte técnico**.
- Con esto la empresa puede **retener** a los clientes en riesgo (ofertas, mejor soporte, contratos más largos).

*Núcleo IA · curso-itec-c-dac.web.app · C-DAC Noida (ITEC) · 2026*